# 6d · E2E entity-type canonicalization  (twin of nb4, for the single-call graph)

Runs the **same** type canonicalization as nb4 — *define → embed → retrieve → reuse-best-match-else-mint* —
on the **e2e** single-call graph, which arrives with **free-form entity types**. **ONE** LLM call per
article covers both kinds of node: each distinct `entities[].type` is matched against
**schema.org ∪ e2e-vocabulary**, and each Event hub's type against the **Event subtree** of the same
pool (so an Event type is always a subtype of `Event`). Entities and hubs are then relabelled **by
id** — ids preserved, so the graph's `relations` stay valid, and NO merging happens, since the single
6c call already resolved coreference. Observation/Claim hub types are left free-form.

- **Config:** one line — `EXPERIMENT`; everything else comes from `configs/<EXPERIMENT>.json`,
  the same file 6c and 6e read.
- **Input:** `results/<model>/<id>/e2e/nary_graph<_sfx>.json` (6c).
- **Output:** `nary_graph<_osfx>_typed.json` (entities/hubs gain `type` = canonical label, plus
  `type_oie`, `type_source ∈ {schema.org, vocab}`, `type_vocab_id`, `type_parent`, …), the two review
  files, and — after the re-consolidation block at the bottom — `nary_graph<_osfx>_typed_consolidated.json`,
  **which is what 6e reads**.
- **Separate vocabulary:** grows `_global/e2e/type_vocabulary<_osfx>.json`, independent of the staged
  track's, so the two methods' type economies stay comparable.

e2e carries per-type definitions from 6c, so the embedding query is `type: definition`, enriched with
synonyms and a few example mentions/aliases. Then run **6e** for the relations.

In [39]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [ ]:
# ---- WHICH EXPERIMENT ---------------------------------------------------------------------
# ONE name decides the whole run. Every knob below comes from configs/<EXPERIMENT>.json — the SAME
# file 6c and 6e read, which is what keeps the three stages of a chain consistent.
#   list what exists :  python -m experiment_config list          (run from notebooks/)
#   see one resolved :  python -m experiment_config show main
# $PHASE2_EXPERIMENT wins when it is set, so a headless batch of ablations (run_e2e.sh) can sweep
# experiments without editing a single notebook.
import os
EXPERIMENT = os.environ.get("PHASE2_EXPERIMENT") or "main"

import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "experiment_config.py").exists():
        sys.path.insert(0, str(_p)); break
import experiment_config as xcfg; importlib.reload(xcfg)

# Sets the shared keys (RESULTS_MODEL, BACKEND, GEMINI_MODEL, MATCH_MODEL, VARIANT, RUN_TAG,
# SELECTED_ARTICLES) and this section's own:
#   FRESH / EXPAND_MODE / TOP_K_MATCH / ALIGN_QUERY_STYLE
#   INLINE_VOCAB_CANDIDATES            which pool the per-article selection call sees
#   RERANK / RERANK_FROM / RERANK_MODEL              inline cross-encoder stage
#   CONSOLIDATE / CONSOLIDATE_TOP_K / CONSOLIDATE_INCLUDE_SCHEMA / SUBTYPE_MODE
#   RERANK_CONSOLIDATE / RERANK_CONSOLIDATE_FROM     consolidation cross-encoder stage
#   INCLUDE_ALL_SCHEMA                 how much of schema.org the printed hierarchy shows
CFG = xcfg.load(EXPERIMENT, globals(), "6d")
# A one-off try goes RIGHT HERE, below the load, where it wins over the file.

# --- imports + derived paths ---------------------------------------------------------------
import json, re, os, copy, time
from pathlib import Path
from collections import Counter
import numpy as np

SCHEMA_DIR  = ROOT / "schema_org_files"          # schema.org CLASS pool from nb1
RESULTS_DIR = ROOT / "results" / RESULTS_MODEL   # reads e2e/nary_graph<_sfx>.json (6c)

# TWO suffixes, and the difference matters:
#   _sfx   = "_" + VARIANT             identifies the 6c extraction arm — used to READ its graphs.
#   _osfx  = _sfx + "_" + RUN_TAG      everything THIS notebook WRITES, so a new configuration
#                                      never overwrites an older run's outputs.
# 6e reads 6d's output, so 6e's INPUT_TAG must equal this RUN_TAG — which it does automatically,
# because both come from the one config file.
_sfx  = f"_{VARIANT}" if VARIANT else ""
_osfx = _sfx + (f"_{RUN_TAG}" if RUN_TAG else "")

VOCAB_PATH = RESULTS_DIR / f"_global/e2e/type_vocabulary{_osfx}.json"   # SEPARATE per run

# ---- RETRIEVAL / RERANK — what the config's knobs mean here --------------------------------
# INLINE_VOCAB_CANDIDATES = False: the TOP_K_MATCH candidates shown to the selection LLM are ALWAYS
# schema.org — the grown vocabulary is never offered inline. The LLM therefore cannot reuse an
# earlier mint, so it re-mints synonyms of terms it already created; collapsing those is deferred to
# the GLOBAL re-consolidation at the bottom. This is 6k's schema-only design with the fix 6k
# concluded it needed: consolidation at GLOBAL scope, not per-article new-vs-prior.
# NOTE the consequence for metrics: the INLINE mint count goes UP by design. Judge this
# configuration on the POST-CONSOLIDATION vocabulary size, not on the raw mint rate.
#
# RERANK and RERANK_CONSOLIDATE are two INDEPENDENT cross-encoder stages, each with its own depth.
# Each depth is that stage's ceiling — the cross-encoder can only promote what e5 already fetched.
# They differ because the pools differ: the inline stage ranks against ~1.5k schema.org classes,
# while consolidation ranks against the vocabulary itself (order 100 entries), so it reaches the
# same coverage far shallower. Cost is linear in the depth on both sides.
# RERANK_MODEL default is 22M params, English, CPU-friendly. Stronger but much slower on CPU:
# "BAAI/bge-reranker-base" (278M) or "BAAI/bge-reranker-v2-m3" (568M, multilingual).

# Which articles to run on — and the ORDER the vocabulary grows along. Order is part of the run:
# this stage is incremental, so what article k mints depends on what k-1 added.
_available = sorted((p.parent.parent.name for p in RESULTS_DIR.glob(f"*/e2e/nary_graph{_sfx}.json")),
                    key=xcfg.article_key)
if SELECTED_ARTICLES is None:
    ARTICLE_ORDER = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print(f"WARNING: skipping (no e2e/nary_graph{_sfx}.json):", missing)
    ARTICLE_ORDER = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_ORDER, (
    f"No runnable articles. Looked for */e2e/nary_graph{_sfx}.json — 6c's output for VARIANT="
    f"{VARIANT!r}. Run 6c under experiment {EXPERIMENT!r} first.")
print(f"\nreads 6c variant {VARIANT!r} -> nary_graph{_sfx}.json")
print(f"writes run       {RUN_TAG!r} -> *{_osfx}*   |  vocab: {VOCAB_PATH.name}")
print(f"{len(ARTICLE_ORDER)} article(s), in this order: {ARTICLE_ORDER}")

In [41]:
# First run only (sentence-transformers pulls torch; google-genai for the LLM stages).
%pip install -q sentence-transformers google-genai numpy

Note: you may need to restart the kernel to use updated packages.


In [42]:
# Embedding model (loaded once). e5 needs "query: " on the query side and "passage: " on documents;
# both the OIE-type queries and every document (schema.org OR vocab) go through the same convention.
from sentence_transformers import SentenceTransformer
_MODEL = SentenceTransformer(MATCH_MODEL)

def _prefixes(name):
    n = name.lower()
    if "e5" in n:  return "query: ", "passage: "
    if "bge" in n: return "Represent this sentence for searching relevant passages: ", ""
    return "", ""
QP, DP = _prefixes(MATCH_MODEL)

def embed(texts, prefix):
    return np.asarray(_MODEL.encode([prefix + t for t in texts],
                                    normalize_embeddings=True, batch_size=64))
print("embedder ready:", MATCH_MODEL, "| query_prefix", repr(QP), "| doc_prefix", repr(DP))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ready: intfloat/e5-base-v2 | query_prefix 'query: ' | doc_prefix 'passage: '


In [43]:
# Schema.org document pool (fixed) — embedded once. Vocabulary documents are embedded per article
# inside retrieval, since the vocabulary grows during the run.
schema_pool   = json.loads((SCHEMA_DIR / "schema_classes_enriched.json").read_text())
SCHEMA_LABELS = {c["label"] for c in schema_pool}
schema_emb    = embed([c["embedding_text"] for c in schema_pool], DP)
print(len(schema_pool), "schema.org classes embedded")

933 schema.org classes embedded


In [44]:
# The growing vocabulary store. One entry per canonical minted type:
#   id, canonical_label, definition, parent (schema.org label or vocab label), aliases[] (= its synonyms),
#   provenance[], support, first_seen, and path (derived: Thing > ... > parent, stamped on every save).
def load_vocab():
    if VOCAB_PATH.exists():
        return json.loads(VOCAB_PATH.read_text())
    return {"types": [], "next_id": 1}

def save_vocab(vocab):
    _annotate_vocab_paths(vocab)                 # refresh each entry's derived `path` from its parent chain
    VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

def vocab_embedding_text(t):
    parent = t.get("parent") or "Thing"
    also   = f" Also called {', '.join(t['aliases'])}." if t.get("aliases") else ""
    return f'{t["canonical_label"]} ({parent}): {t.get("definition", "")}{also}'

def _find_vocab_by_label(vocab, label):
    for t in vocab["types"]:
        if t["canonical_label"] == label or label in t.get("aliases", []):
            return t
    return None

def _is_vocab_label(vocab, label):
    return _find_vocab_by_label(vocab, label) is not None

def _bump(ent, aid, oie_type, group):
    ent["support"] = ent.get("support", 0) + 1
    ent.setdefault("provenance", []).append(
        {"article": aid, "oie_type": oie_type, "mentions": [e["text"] for e in group]})

In [ ]:
# --- Cross-encoder reranker (no training) -------------------------------------------------------
# Twin of 6e's reranker, for TYPES. e5 compares a query vector and a class vector built
# independently — it never sees the two together, so a class whose definition happens to be phrased
# like the query can outrank the one that actually means the same thing. A cross-encoder reads the
# pair JOINTLY and scores the match directly. It is far too slow to run over the whole class pool
# per type, so it goes second:
#
#     e5 retrieves RERANK_FROM (cheap, high recall)  ->  cross-encoder cuts to the ten the LLM sees
#
# RERANK / RERANK_FROM / RERANK_MODEL come from the experiment config — do NOT redefine them here,
# or the cell would silently override the configuration the run is supposed to be reproducing.
_RERANKER = None

def _reranker():
    """Loaded on first use, so the notebook costs nothing while RERANK is off."""
    global _RERANKER
    if _RERANKER is None:
        from sentence_transformers import CrossEncoder
        print(f"loading reranker {RERANK_MODEL} ...")
        _RERANKER = CrossEncoder(RERANK_MODEL)
    return _RERANKER

def rerank_candidates(query, candidates, top_k):
    """Re-order ONE type's candidates by cross-encoder relevance, then cut to top_k. The document
    side is "label: definition" — the definition is what separates schema.org's near-siblings.
    `score` becomes the cross-encoder score (a logit: it can be negative and is NOT comparable to a
    cosine); the retrieval score and rank are kept as `embed_score` / `embed_rank`."""
    if not candidates:
        return []
    pairs  = [(query, f'{c["label"]}: {(c.get("comment") or "")[:400]}') for c in candidates]
    scores = np.asarray(_reranker().predict(pairs, batch_size=64, show_progress_bar=False),
                        dtype="float64")
    out = []
    for j in np.argsort(-scores)[:top_k]:
        c = dict(candidates[int(j)])
        c["embed_score"] = c["score"]
        c["embed_rank"]  = int(j) + 1
        c["score"]       = round(float(scores[int(j)]), 4)
        out.append(c)
    return out

print(f"reranker ready (lazy) | RERANK={RERANK} | {RERANK_MODEL}")
print(f"  outputs -> *{_osfx}*   |  vocab: {VOCAB_PATH.name}")


In [46]:
# Hook A — retrieve top-k candidates for each OIE type. The pool is schema.org, plus the grown
# vocabulary ONLY when INLINE_VOCAB_CANDIDATES is on (config cell). With it off the LLM never sees an
# earlier mint here, so it cannot reuse one — re-minted synonyms are collapsed later by the global
# re-consolidation instead. RERANK (config cell) adds a cross-encoder pass over the shortlist.
# Both schema.org AND vocab candidates expose a full ancestor `path` (Thing > ... > immediate parent),
# so the type-selection LLM can reach for any ancestor as a mint parent, symmetrically for either source.
_SCHEMA_BY_LABEL = {c["label"]: c for c in schema_pool}

def _schema_chain(label):
    """Top-down ancestor chain INCLUDING `label` for a schema.org class: [Thing, ..., label].
    (schema.org `path` lists ancestors only — its last element is the immediate parent.)"""
    c   = _SCHEMA_BY_LABEL.get(label)
    anc = [a for a in (c.get("path", "").split(" > ") if c and c.get("path") else []) if a]
    return anc + [label]

def vocab_ancestor_path(t, vocab):
    """Full ancestor path for a vocab type, formatted exactly like schema.org's `path` (top-down,
    ending at the immediate parent, EXCLUDING the type itself). Climbs parent links up through the
    vocabulary, then splices in the schema.org chain of the first schema ancestor it reaches. A
    `seen` guard keeps a bad re-parent (e.g. from consolidation) from looping forever."""
    up, cur, seen = [], t.get("parent"), set()
    while cur and cur not in seen:
        seen.add(cur)
        ent = _find_vocab_by_label(vocab, cur)
        if ent:                                   # another vocab type -> keep climbing the vocab tree
            up.append(cur); cur = ent.get("parent")
        else:                                     # a schema.org label (or Thing) -> splice its chain in
            up.extend(reversed(_schema_chain(cur))); cur = None
    return " > ".join(reversed(up))

def _annotate_vocab_paths(vocab):
    """Stamp each entry's derived `path` (Thing > ... > immediate parent), matching schema.org's `path`
    format. Derived from the `parent` chains (the source of truth) — refreshed on every save, so it
    stays correct even after consolidation re-parents a type."""
    for t in vocab["types"]:
        t["path"] = vocab_ancestor_path(t, vocab)
    return vocab

def _full_path(tref, vocab):
    """Full ancestor path for a resolved type reference (used in the review's chosen_type block):
    a vocab entry's parent chain, or a schema.org class's ancestor chain; None otherwise."""
    if tref.get("source") == "vocab" and tref.get("vocab_id"):
        ent = next((t for t in vocab["types"] if t["id"] == tref["vocab_id"]), None)
        if ent is not None:
            return vocab_ancestor_path(ent, vocab)
    if tref.get("source") == "schema.org" and tref.get("label"):
        return " > ".join(_schema_chain(tref["label"])[:-1])   # ancestors only (exclude the class itself)
    return None

def retrieve_candidates(type_texts, vocab, top_k=None, rerank=None):
    """Top-k candidate classes per OIE type. With `rerank` (default: the RERANK flag) it fetches
    RERANK_FROM deep and cross-encodes down to top_k, so flipping RERANK changes the real build."""
    top_k  = TOP_K_MATCH if top_k is None else top_k
    rerank = RERANK if rerank is None else rerank
    fetch  = max(top_k, RERANK_FROM) if rerank else top_k
    q_emb = embed(type_texts, QP)
    meta  = [("schema.org", c) for c in schema_pool]
    d_emb = schema_emb
    if INLINE_VOCAB_CANDIDATES and vocab["types"]:
        v_emb = embed([vocab_embedding_text(t) for t in vocab["types"]], DP)
        d_emb = np.vstack([schema_emb, v_emb])
        meta  = meta + [("vocab", t) for t in vocab["types"]]
    sims = q_emb @ d_emb.T
    topk = np.argsort(-sims, axis=1)[:, :fetch]
    out = []
    for ti in range(len(type_texts)):
        row = []
        for j in topk[ti]:
            src, obj = meta[j]
            if src == "schema.org":
                row.append({"source": "schema.org", "label": obj["label"], "path": obj.get("path", ""),
                            "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("enriched_comment") or obj.get("comment", "")})
            else:
                row.append({"source": "vocab", "id": obj["id"], "label": obj["canonical_label"],
                            "path": vocab_ancestor_path(obj, vocab), "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("definition", "")})
        out.append(row)
    if rerank:      # second stage: cross-encode the deep list down to the ten the LLM sees
        out = [rerank_candidates(q, row, top_k) for q, row in zip(type_texts, out)]
    return out

In [ ]:
# Gemini client (JSON mode, temperature 0 for reproducibility).
from google import genai
from google.genai import types as genai_types
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in the env and restart the kernel."
client = genai.Client(api_key=GEMINI_API_KEY)

def gemini_json(system, user):
    import time
    t0 = time.perf_counter()
    r = client.models.generate_content(
        model=GEMINI_MODEL, contents=user,
        config=genai_types.GenerateContentConfig(
            system_instruction=system, temperature=0, response_mime_type="application/json"))
    global LAST_USAGE
    LAST_USAGE = nb_parallel.record_gemini_usage(r, time.perf_counter() - t0)   # prompt/output/THINKING tokens + latency
    return r.text

def parse_json(text):
    # The two backends disagree on what they hand back: the Gemini client above returns r.text (a
    # STRING), while llm_backend.llm_json already runs the notebooks' own lenient loader and returns
    # a PARSED object. Accepting either keeps one call site working on both, instead of this stage
    # dying on `'dict' object has no attribute 'strip'` the moment the backend is an open model.
    if isinstance(text, (dict, list)):
        return text
    s = (text or "").strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\s*", "", s); s = re.sub(r"\s*```$", "", s)
    return json.loads(s)


# --- select the backend ---------------------------------------------------------------------
# Must stay LAST in this cell: the Gemini `gemini_json` above is redefined every time the cell
# runs, so whichever binding happens last wins. In a separate cell this would be silently undone.
# activate() sets gemini_json, GEMINI_MODEL, RESULTS_DIR, MAX_OUTPUT_TOKENS and THINKING_BUDGET.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend
importlib.reload(nb_parallel); importlib.reload(llm_backend)   # plain `import` is a no-op once cached
nb_parallel.install()           # thread-aware stdout + the token/latency ledger used by usage_report

llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)


In [ ]:
# ---- Minting: admit a chosen "new" type into the vocabulary -----------------------------------
# No mint-time canonicalization — the mint is admitted as-is. Collapsing synonymous / broader mints
# onto one entry is handled later by the OPTIONAL global consolidate_vocab pass.
def _schema_ref(label):
    return {"label": label, "source": "schema.org", "vocab_id": None,
            "definition": None, "parent": None, "is_new_to_vocab": False}

def _vocab_ref(ent, is_new=False):
    return {"label": ent["canonical_label"], "source": "vocab", "vocab_id": ent["id"],
            "definition": ent["definition"], "parent": ent["parent"], "is_new_to_vocab": is_new}

# Admit a minted type as a NEW vocabulary entry (or, in restrict mode, fall back to its parent).
def _apply_mint(aid, oie_type, ch, group, vocab):
    label = ch.get("label"); definition = ch.get("definition") or ""
    existing = _find_vocab_by_label(vocab, label)          # dedup by label: never mint a duplicate entry
    if existing is not None:                               # (the LLM may say is_new for a label already in vocab)
        _bump(existing, aid, oie_type, group)
        return _vocab_ref(existing)
    parent = ch.get("parent") or "Thing"
    if not EXPAND_MODE:                                        # restrict mode: fall back to the parent
        return _schema_ref(parent)
    if not (parent in SCHEMA_LABELS or _is_vocab_label(vocab, parent)):
        parent = "Thing"
    ent = {"id": f"T{vocab['next_id']:04d}", "canonical_label": label, "definition": definition,
           "parent": parent, "aliases": [], "support": 1, "first_seen": aid,
           "provenance": [{"article": aid, "oie_type": oie_type, "mentions": [e["text"] for e in group]}]}
    vocab["types"].append(ent); vocab["next_id"] += 1
    return _vocab_ref(ent, is_new=True)


In [ ]:
# ── MERGED type canonicalization: ONE Gemini call per article for BOTH entity types AND Event-hub types.
# Replaces the old two-pass approach (entity Build + a separate Event-hub pass = 2 calls/article) with a
# single call/article (~50% fewer calls). Each group carries its OWN retrieved CANDIDATES, so entity
# types still match the FULL schema.org pool while Event types match only the Event subtree; the "every
# Event type is a subtype of Event" guarantee is enforced at retrieval + mint (per KIND), NOT by the calls
# being separate. Both kinds grow the SAME _global/e2e/type_vocabulary.json.
# Selection-only: the old within-group dedup (TASK 2) is dropped — the single 6c call already resolved
# coreference and 6d never used those clusters (it relabels entities by id, no merge).

EVENT_ROOT = "Event"

def _under_event_schema(label):
    return EVENT_ROOT in _schema_chain(label)                       # _schema_chain includes `label`

def _under_event_vocab(t, vocab):
    return t["canonical_label"] == EVENT_ROOT or \
           EVENT_ROOT in vocab_ancestor_path(t, vocab).split(" > ")

def retrieve_event_candidates(type_texts, vocab):
    """Top-k retrieval like retrieve_candidates, but the pool is restricted to the Event subtree
    (schema.org Event-subtree classes u vocab types parented under Event). Like retrieve_candidates,
    the vocabulary half is offered ONLY when INLINE_VOCAB_CANDIDATES is on — otherwise BOTH kinds of
    group see schema.org only, and the schema-only prompt below tells the truth about its input."""
    sch  = [c for c in schema_pool if _under_event_schema(c["label"])]
    meta = [("schema.org", c) for c in sch]
    docs = [c["embedding_text"] for c in sch]
    if INLINE_VOCAB_CANDIDATES:
        for t in vocab["types"]:
            if _under_event_vocab(t, vocab):
                meta.append(("vocab", t)); docs.append(vocab_embedding_text(t))
    if not docs:
        return [[] for _ in type_texts]
    d_emb = embed(docs, DP); q_emb = embed(type_texts, QP)
    sims  = q_emb @ d_emb.T
    topk  = np.argsort(-sims, axis=1)[:, :TOP_K_MATCH]
    out = []
    for ti in range(len(type_texts)):
        row = []
        for j in topk[ti]:
            src, obj = meta[j]
            if src == "schema.org":
                row.append({"source": "schema.org", "label": obj["label"], "path": obj.get("path", ""),
                            "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("enriched_comment") or obj.get("comment", "")})
            else:
                row.append({"source": "vocab", "id": obj["id"], "label": obj["canonical_label"],
                            "path": vocab_ancestor_path(obj, vocab), "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("definition", "")})
        out.append(row)
    return out

def _bump_hub(ent, aid, etype, group):
    ent["support"] = ent.get("support", 0) + 1
    ent.setdefault("provenance", []).append(
        {"article": aid, "oie_type": etype, "mentions": [h.get("label") for h in group]})

def _apply_mint_hub(aid, etype, ch, group, vocab):
    """Admit a minted Event type; parent is forced into the Event subtree (default 'Event', never 'Thing')."""
    parent, label = ch.get("parent") or EVENT_ROOT, ch.get("label")
    existing = _find_vocab_by_label(vocab, label)          # dedup by label: never mint a duplicate entry
    if existing is not None:
        _bump_hub(existing, aid, etype, group)
        return _vocab_ref(existing)
    ok = (parent in SCHEMA_LABELS and _under_event_schema(parent))
    if not ok:
        pe = _find_vocab_by_label(vocab, parent)
        ok = bool(pe and _under_event_vocab(pe, vocab))
    if not ok:
        parent = EVENT_ROOT
    if not EXPAND_MODE:
        return _schema_ref(parent)
    ent = {"id": f"T{vocab['next_id']:04d}", "canonical_label": label, "definition": ch.get("definition") or "",
           "parent": parent, "aliases": [], "support": 1, "first_seen": aid,
           "provenance": [{"article": aid, "oie_type": etype, "mentions": [h.get("label") for h in group]}]}
    vocab["types"].append(ent); vocab["next_id"] += 1
    return _vocab_ref(ent, is_new=True)

# ==== EDIT ME — unified selection prompts (entity + event groups in one call, selection only) =======
# TWO variants, picked at call time by INLINE_VOCAB_CANDIDATES (config cell) so the prompt always
# describes the candidates the model actually receives:
#   MIXED  (INLINE_VOCAB_CANDIDATES = True)  — candidates are schema.org classes AND earlier mints,
#          marked [schema.org] / [vocab]; the model may reuse a mint, so `source` distinguishes them.
#   SCHEMA (INLINE_VOCAB_CANDIDATES = False) — candidates are schema.org ONLY. The vocabulary is
#          deliberately hidden, so the prompt tells the model not to guess at unseen mints and that
#          re-minting across documents is expected (the global re-consolidation collapses duplicates).
TYPE_SELECT_PROMPT_MIXED = """You are canonicalizing free-form types against a shared type vocabulary that is reused across many documents.
You are given several GROUPS from one news document. Each group is a single free-form OIE type to canonicalize, and has one of two kinds: ENTITY (a type for real-world entities: people, organizations, places, quantities, ...) or EVENT (a type for a kind of event).

Each group provides:
- KIND: ENTITY or EVENT.
- OIE_TYPE: the free-form type label (plus its definition, if any).
- CANDIDATES: a ranked list of existing types proposed as matches — each a standard schema.org class marked [schema.org], or a type minted for an earlier document marked [vocab] — with its comment and its ancestor `path`. Every label shown, and every ancestor in a `path`, is available both for reuse and as a parent.
- MENTIONS / EXAMPLES: the surface strings (ENTITY) or event titles (EVENT) that were given this type.

For each group independently — never compare or merge across groups — choose the single best type:
- Reuse the candidate whose meaning best fits OIE_TYPE and the mentions/examples. One that is somewhat broader or narrower still beats minting, provided it preserves what makes the type distinctive; between two that fit equally well, take the [schema.org] one.
- Instance test: judge by the MENTIONS/EXAMPLES, not the definition alone — would each individually be a valid instance of the candidate? If they are open-ended categories, roles, or kinds rather than instances of it, it is too broad; reject it.
- Mint only when no candidate passes the instance test: a concise, reusable PascalCase label with no specific named entity in it, a one-sentence definition, and as parent the closest listed type it is a subclass of.
- When KIND = EVENT, the chosen type (reused or minted) must be a subtype of schema.org Event, and a minted one's parent must be "Event", a more specific Event-subtree class, or a [vocab] event type — never "Thing". When KIND = ENTITY, use "Thing" only as a last resort.

Return only valid JSON, one object per group, in the input order:
{"results": [{"id": "<group id>", "chosen_type": {"source": "schema.org | vocab | new", "label": "<type label>", "is_new": false, "definition": null, "parent": null, "justification": "one short line"}}]}
When you reuse, `source` echoes that candidate's marked source and `definition` / `parent` stay null; when you mint, set `source` to "new" and `is_new` true, and fill both."""

TYPE_SELECT_PROMPT_SCHEMA = """You are canonicalizing free-form types against schema.org, minting a new type only where no schema.org class fits.
You are given several GROUPS from one news document. Each group is a single free-form OIE type to canonicalize, and has one of two kinds: ENTITY (a type for real-world entities: people, organizations, places, quantities, ...) or EVENT (a type for a kind of event).

Each group provides:
- KIND: ENTITY or EVENT.
- OIE_TYPE: the free-form type label (plus its definition, if any).
- CANDIDATES: a ranked list of schema.org classes proposed as matches, each with its comment and its ancestor `path`. Every class shown, and every ancestor in a `path`, is available both for reuse and as a parent.
- MENTIONS / EXAMPLES: the surface strings (ENTITY) or event titles (EVENT) that were given this type.

For each group independently — never compare or merge across groups — choose the single best type:
- Reuse the candidate whose meaning best fits OIE_TYPE and the mentions/examples. One that is somewhat broader or narrower still beats minting, provided it preserves what makes the type distinctive.
- Instance test: judge by the MENTIONS/EXAMPLES, not the definition alone — would each individually be a valid instance of the candidate? If they are open-ended categories, roles, or kinds rather than instances of it, it is too broad; reject it.
- Mint only when no candidate passes that test: a concise, reusable PascalCase label with no specific named entity in it, a one-sentence definition, and as parent the closest listed class it is a subclass of.
- When KIND = EVENT, the chosen type (reused or minted) must be a subtype of schema.org Event, and a minted one's parent must be "Event" or a more specific Event-subtree class. When KIND = ENTITY, use "Thing" only as a last resort.

Return only valid JSON, one object per group, in the input order:
{"results": [{"id": "<group id>", "chosen_type": {"source": "schema.org | new", "label": "<type label>", "is_new": false, "definition": null, "parent": null, "justification": "one short line"}}]}
When you reuse, `source` is "schema.org" and `definition` / `parent` stay null; when you mint, set `source` to "new" and `is_new` true, and fill both."""

def active_select_prompt():
    """The prompt matching the candidates retrieval actually produces (see INLINE_VOCAB_CANDIDATES)."""
    return TYPE_SELECT_PROMPT_MIXED if INLINE_VOCAB_CANDIDATES else TYPE_SELECT_PROMPT_SCHEMA

def llm_select_types(items):
    """ONE Gemini call over ENTITY + EVENT groups. `items`: dicts with gid/kind/oie_type/definition/
    examples/candidates. Returns {gid: chosen_type}. Missing groups fall back to the top candidate."""
    blocks = []
    for it in items:
        b = [f'GROUP {it["gid"]} — KIND: {it["kind"].upper()}', f'OIE_TYPE: "{it["oie_type"]}"']
        if it.get("definition"):
            b.append(f'DEFINITION: {it["definition"]}')
        b.append("CANDIDATES (ranked):")
        if it["candidates"]:
            for k, c in enumerate(it["candidates"], 1):
                b.append(f'  {k}. [{c["source"]}] {c["label"]} | path/parent: {c.get("path") or "(datatype)"}')
                b.append(f'     {c.get("comment", "")}')
        else:
            b.append("  (none)")
        b.append("EXAMPLES:" if it["kind"] == "event" else "MENTIONS:")
        b += [f'  - "{x}"' for x in (it["examples"] or [])[:6]]
        blocks.append("\n".join(b))
    user = f"You are given {len(items)} GROUPS from one document.\n\n" + "\n\n".join(blocks)
    parsed  = parse_json(gemini_json(active_select_prompt(), user))
    results = parsed.get("results", parsed) if isinstance(parsed, dict) else parsed
    # An open-weight model in SOFT-JSON mode (thinking arms: no grammar constraint) sometimes
    # answers `"chosen_type": "Person"` — a bare string where the schema says an object — and
    # occasionally drops a bare string into `results` too. `or {}` kept the string, which then died
    # ~200 lines later on `ch.get(...)`, after the whole chunk's LLM calls had already been paid for.
    # Coerce at the parse boundary: a bare label is exactly what the no-match fallback path expects,
    # and a malformed row is skipped rather than being fatal.
    def _as_choice(v):
        if isinstance(v, str):
            return {"label": v}
        return v if isinstance(v, dict) else {}

    out = {r.get("id"): _as_choice(r.get("chosen_type"))
           for r in (results or []) if isinstance(r, dict) and r.get("id")}
    for it in items:                                          # fallback: omitted group -> top candidate
        if it["gid"] not in out:
            top = it["candidates"][0] if it["candidates"] else \
                  {"source": "schema.org", "label": EVENT_ROOT if it["kind"] == "event" else "Thing"}
            out[it["gid"]] = {"source": top["source"], "label": top["label"], "is_new": False,
                              "justification": "fallback: top candidate"}
    return out

def prefetch_types(aid, vocab):
    """PHASE 1 — everything up to AND INCLUDING the LLM call, mutating nothing.

    `vocab` is read only by retrieve_candidates / retrieve_event_candidates, and both ignore it while
    INLINE_VOCAB_CANDIDATES is False — which is exactly what makes these calls independent, and so
    safe to run in parallel. The moment the vocabulary is offered inline, article k's candidates
    depend on what k-1 minted and this may no longer be prefetched (the build cell checks).

    Token usage needs no special handling: pmap opens a usage_scope per item and the ledger is
    thread-local, so each article keeps its own counts with several calls in flight.
    """
    g = json.loads((RESULTS_DIR / aid / f"e2e/nary_graph{_sfx}.json").read_text())
    tinfo = {d["type"]: d for d in g.get("type_definitions", [])}

    # ENTITY groups — query text mirrors load_article's qtext ------------------------------------
    ent_types  = list(dict.fromkeys(e["type"] for e in g["entities"]))
    ent_groups = {t: [e for e in g["entities"] if e["type"] == t] for t in ent_types}
    def _eq(t):
        info = tinfo.get(t, {})
        q = f'{t}: {info["definition"]}' if info.get("definition") else t
        if ALIGN_QUERY_STYLE and info.get("synonyms"):
            q = f'{q} Also called {", ".join(info["synonyms"])}.'
        ex = [e.get("text") for e in ent_groups[t]] + [a for e in ent_groups[t] for a in (e.get("aliases") or [])]
        ex = [x for x in dict.fromkeys(ex) if x][:4]
        if ALIGN_QUERY_STYLE and ex:
            q = f'{q} Examples: {", ".join(ex)}.'
        return q
    ent_q     = [_eq(t) for t in ent_types]
    ent_cands = retrieve_candidates(ent_q, vocab)

    # EVENT hub groups (node_kind == "Event"; their `type` is still the 6c free-form label) -------
    ev          = [n for n in g.get("nary_nodes", []) if n.get("node_kind") == EVENT_ROOT]
    ev_types    = list(dict.fromkeys(n.get("type") for n in ev))
    ev_groups   = {t: [n for n in ev if n.get("type") == t] for t in ev_types}
    ev_examples = {t: [n.get("label", "") for n in ev_groups[t]] for t in ev_types}
    ev_q = []
    for t in ev_types:
        ex = ", ".join(x for x in ev_examples[t][:4] if x)
        ev_q.append(f'{t} (examples: {ex})' if ex else t)
    ev_cands = retrieve_event_candidates(ev_q, vocab)

    # ONE combined LLM call ---------------------------------------------------------------------
    items  = [{"gid": f"E{i+1}", "kind": "entity", "oie_type": t, "definition": tinfo.get(t, {}).get("definition"),
               "examples": [e["text"] for e in ent_groups[t]], "candidates": ent_cands[i]}
              for i, t in enumerate(ent_types)]
    items += [{"gid": f"V{i+1}", "kind": "event", "oie_type": t, "definition": None,
               "examples": ev_examples[t], "candidates": ev_cands[i]}
              for i, t in enumerate(ev_types)]
    report = llm_select_types(items) if items else {}
    return {"g": g, "ent_types": ent_types, "ent_groups": ent_groups, "ent_cands": ent_cands,
            "ent_q": ent_q, "ev_types": ev_types, "ev_groups": ev_groups,
            "ev_examples": ev_examples, "ev_cands": ev_cands, "items": items, "report": report}


def canonicalize_article_types(aid, vocab, write=True, save=True, verbose=True, pre=None):
    """PHASE 2 — apply one article's selections to the vocabulary, in ARTICLE_ORDER.

    Relabels entities + Event hubs BY ID (relations stay valid), grows `vocab`, and writes
    nary_graph{_osfx}_typed.json + entity/hub review files. Observation/Claim hub types stay
    free-form. `pre` is a prefetch_types() result; without one the call happens here, so the
    sequential path is unchanged.

    Everything BELOW this point mutates the vocabulary and therefore stays ordered: what article k
    can reuse depends on what k-1 added, which is the whole reason the two phases are separate.
    """
    if pre is None:
        pre = prefetch_types(aid, vocab)
    g, report, items = pre["g"], pre["report"], pre["items"]
    ent_types, ent_groups, ent_cands, ent_q = (pre["ent_types"], pre["ent_groups"],
                                               pre["ent_cands"], pre["ent_q"])
    ev_types, ev_groups, ev_examples, ev_cands = (pre["ev_types"], pre["ev_groups"],
                                                  pre["ev_examples"], pre["ev_cands"])

    # Resolve ENTITY types (mint parents anywhere; Thing fallback) -------------------------------
    ent_res, n_mint_e = {}, 0
    for i, t in enumerate(ent_types):
        ch = report.get(f"E{i+1}", {})
        if ch.get("is_new") or ch.get("source") == "new":
            before = len(vocab["types"]); tref = _apply_mint(aid, t, ch, ent_groups[t], vocab)
            n_mint_e += (len(vocab["types"]) > before)
        else:
            ent = _find_vocab_by_label(vocab, ch.get("label"))
            if ent: _bump(ent, aid, t, ent_groups[t]); tref = _vocab_ref(ent)
            else:   tref = _schema_ref(ch.get("label"))
        tref["justification"] = ch.get("justification"); ent_res[t] = tref

    # Resolve EVENT types (mint parents forced under Event) --------------------------------------
    ev_res, n_mint_v = {}, 0
    for i, t in enumerate(ev_types):
        ch = report.get(f"V{i+1}", {})
        if ch.get("is_new") or ch.get("source") == "new":
            before = len(vocab["types"]); tref = _apply_mint_hub(aid, t, ch, ev_groups[t], vocab)
            n_mint_v += (len(vocab["types"]) > before)
        else:
            ent = _find_vocab_by_label(vocab, ch.get("label"))
            if ent: _bump_hub(ent, aid, t, ev_groups[t]); tref = _vocab_ref(ent)
            else:   tref = _schema_ref(ch.get("label"))
        tref["justification"] = ch.get("justification"); ev_res[t] = tref

    # Relabel entities BY ID (preserve id/text/aliases; NO merge) --------------------------------
    ents = []
    for e in g["entities"]:
        tref = ent_res[e["type"]]
        ents.append({**e, "type": tref.get("label"), "type_oie": e["type"],
                     "type_source": tref.get("source"), "type_vocab_id": tref.get("vocab_id"),
                     "type_is_new_to_vocab": tref.get("is_new_to_vocab", False),
                     "type_definition": tref.get("definition"), "type_parent": tref.get("parent"),
                     "type_justification": tref.get("justification")})

    # Relabel Event hubs BY ID (keep type_oie); leave Observation/Claim types free-form ----------
    nary = [dict(n) for n in g.get("nary_nodes", [])]
    for n in nary:
        if n.get("node_kind") == EVENT_ROOT:
            t = n.get("type"); tref = ev_res[t]
            n.update({"type": tref.get("label"), "type_oie": t, "type_source": tref.get("source"),
                      "type_vocab_id": tref.get("vocab_id"),
                      "type_is_new_to_vocab": tref.get("is_new_to_vocab", False),
                      "type_definition": tref.get("definition"), "type_parent": tref.get("parent"),
                      "type_justification": tref.get("justification")})

    # Refresh relation endpoint types from entities + all n-ary nodes ----------------------------
    id2type = {e["id"]: e["type"] for e in ents}
    id2type.update({n["id"]: n.get("type") for n in nary})
    rels, _bad = [], 0
    for r in g.get("relations", []):
        if not r.get("subject_id") or not r.get("object_id"):    # drop malformed half-edges
            _bad += 1; continue
        rels.append({**r, "subject_type": id2type.get(r["subject_id"], r.get("subject_type")),
                          "object_type":  id2type.get(r["object_id"],  r.get("object_type"))})
    if _bad:
        print(f"    [warn] {aid}: dropped {_bad} relation(s) missing subject_id/object_id")

    if write:
        out_dir = RESULTS_DIR / aid / "e2e"
        (out_dir / f"nary_graph{_osfx}_typed.json").write_text(json.dumps(
            {**{k: g[k] for k in ("article_id", "reference_date", "method") if k in g},
             "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE,
             "entities": ents, "nary_nodes": nary,
             "relations": rels, "relation_definitions": g.get("relation_definitions", [])},
            indent=2, ensure_ascii=False))
        ent_review = [{
            "oie_type": t, "embedding_query_text": ent_q[i],
            "chosen_type": {"label": ent_res[t].get("label"), "source": ent_res[t].get("source"),
                "vocab_id": ent_res[t].get("vocab_id"), "is_new_to_vocab": ent_res[t].get("is_new_to_vocab", False),
                "definition": ent_res[t].get("definition"), "parent": ent_res[t].get("parent"),
                "path": _full_path(ent_res[t], vocab), "justification": ent_res[t].get("justification")},
            "llm_proposed": {k: (report.get(f"E{i+1}", {}) or {}).get(k) for k in ("source", "label", "is_new")},
            "mentions": [e["text"] for e in ent_groups[t]], "candidates": ent_cands[i]}
            for i, t in enumerate(ent_types)]
        (out_dir / f"entity_type_matching_review{_osfx}.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE, "types": ent_review},
            indent=2, ensure_ascii=False))
        (out_dir / f"hub_type_matching_review{_osfx}.json").write_text(json.dumps(
            {"article_id": aid, "types": [{"event_type": t, "chosen": ev_res[t],
              "examples": ev_examples[t], "candidates": ev_cands[i]} for i, t in enumerate(ev_types)]},
            indent=2, ensure_ascii=False))
    if save:
        save_vocab(vocab)
    if verbose:
        nvoc = sum(1 for e in ents if e.get("type_source") == "vocab")
        print(f"  {aid}: {len(ent_types)} entity + {len(ev_types)} event types over {len(ents)} entities "
              f"| +{n_mint_e + n_mint_v} minted | {nvoc} ent via vocab | vocab now {len(vocab['types'])}")
    return ents, nary, vocab

print("type-selection prompt:", "MIXED (schema.org + vocab)" if INLINE_VOCAB_CANDIDATES
      else "SCHEMA-ONLY (vocabulary hidden from the LLM)")


## Build — incremental vocabulary over all articles (in order)

In [ ]:
# Incremental build — ONE LLM call per article canonicalizes BOTH entity types AND Event-hub types.
# Sequential BY DESIGN: the vocabulary grows article by article and what article k mints depends on
# what k-1 added, so running the articles in parallel would change the output, not just the speed.
#
# FRESH comes from the config. True: start from an EMPTY vocabulary and process ALL of
# ARTICLE_ORDER (what a reported run does — reproducible from the config alone).
# False: keep the existing type_vocabulary<_osfx>.json and extend it with NEW_ARTICLES only
# (or, if that is empty, with every article that has no typed graph yet).
NEW_ARTICLES = []          # used only when FRESH is False

if FRESH:
    vocab = {"types": [], "next_id": 1}; save_vocab(vocab); todo = ARTICLE_ORDER
else:
    vocab = load_vocab()
    todo  = NEW_ARTICLES or [a for a in ARTICLE_ORDER
                             if not (RESULTS_DIR / a / f"e2e/nary_graph{_osfx}_typed.json").exists()]

# CONSOLIDATE_EVERY (config, 0 = off) re-consolidates the vocabulary every N articles instead of
# once at the end. This cell then builds only the FIRST chunk: consolidate_vocab and
# refresh_ext_after_consolidation are defined further down, so the remaining chunks are driven by
# the periodic cell BELOW them. With the knob at 0 this is exactly the old behaviour.
TODO_ALL = list(todo)
if CONSOLIDATE_EVERY:
    todo = TODO_ALL[:CONSOLIDATE_EVERY]

print(f"building e2e type vocabulary over {len(todo)} article(s): {todo}\n")
nb_parallel.reset_usage()          # the report below covers only the calls THIS run makes
_t0 = time.perf_counter()

# PARALLEL_LLM overlaps the per-article calls (phase 1) and then applies them in ARTICLE_ORDER
# (phase 2). Valid ONLY while the inline pool is schema.org-only: with the vocabulary offered inline
# each article's candidates depend on the previous articles' mints, so the calls stop being
# independent and the run would differ, not just go faster.
_pre = {}
if PARALLEL_LLM and INLINE_VOCAB_CANDIDATES:
    print("PARALLEL_LLM ignored: INLINE_VOCAB_CANDIDATES is True, so each article's candidates\n"
          "depend on the vocabulary the earlier articles built — the calls are NOT independent.\n")
elif PARALLEL_LLM and len(todo) > 1:
    print(f"phase 1 — {len(todo)} LLM calls on {WORKERS} threads (inline pool is schema.org only)")
    _pre = {r.item: r.value for r in
            nb_parallel.pmap(lambda a: prefetch_types(a, vocab), todo, workers=WORKERS)
            if r.ok}
    print("\nphase 2 — applying to the vocabulary in ARTICLE_ORDER (no LLM calls)")

for aid in todo:
    with nb_parallel.usage_scope(aid):      # tag this article's calls so the report breaks down per article
        canonicalize_article_types(aid, vocab, write=True, save=True, verbose=True, pre=_pre.get(aid))
    if aid not in _pre:                     # nothing to pace when the call already happened
        time.sleep(5)                       # gentle on the API's per-minute limit
print(f"\ndone — {len(vocab['types'])} canonical types in {VOCAB_PATH}")

# --- what this run cost: calls, prompt / output / THINKING tokens, and elapsed time ------------
# Written to a tagged file so each experiment keeps its own cost record (run_metrics reads it).
nb_parallel.usage_report(f"6d · e2e type canonicalization · {EXPERIMENT}",
                         wall_seconds=time.perf_counter() - _t0,
                         write=RESULTS_DIR / "_global" / "e2e" / f"token_usage_types{_osfx}.json",
                         experiment=EXPERIMENT, run_tag=RUN_TAG)

## Report + verify

In [53]:
# e2e type-vocabulary summary, then verify each typed graph preserves its input entity ids.
vocab = load_vocab()
print(f"E2E TYPE VOCABULARY — {len(vocab['types'])} canonical types\n" + "-" * 74)
for t in sorted(vocab["types"], key=lambda x: -x.get("support", 0)):
    print(f"  {t['id']}  {t['canonical_label']:<24} <: {str(t.get('parent')):<16} support={t.get('support')}")

print("\nVERIFY typed graph vs e2e/nary_graph.json (entity ids preserved)\n" + "-" * 74)
ok = True
for aid in ARTICLE_ORDER:
    tf = RESULTS_DIR / aid / f"e2e/nary_graph{_osfx}_typed.json"
    if not tf.exists():
        print(f"  {aid:<8} (no typed file)"); continue
    raw = json.loads((RESULTS_DIR / aid / f"e2e/nary_graph{_sfx}.json").read_text())
    typ = json.loads(tf.read_text())
    ids_raw = sorted(e["id"] for e in raw["entities"])
    ids_typ = sorted(e["id"] for e in typ["entities"])
    nvoc = sum(1 for e in typ["entities"] if e.get("type_source") == "vocab")
    same = ids_raw == ids_typ; ok &= same
    print(f"  {aid:<8} {len(ids_raw):>4} entities  ids_preserved={same}  ({nvoc} typed via vocab)")
print("-" * 74); print("consistent" if ok else "INCONSISTENCIES")

E2E TYPE VOCABULARY — 86 canonical types
--------------------------------------------------------------------------
  T0001  DateRange                <: StructuredValue  support=3
  T0002  Month                    <: Enumeration      support=2
  T0003  GovernmentOfficial       <: Person           support=2
  T0004  AgeGroup                 <: Enumeration      support=1
  T0005  Temperature              <: Quantity         support=1
  T0006  AlertLevel               <: StatusEnumeration support=1
  T0007  WeatherSystem            <: Thing            support=1
  T0008  Archipelago              <: Landform         support=1
  T0009  HolidayWeekend           <: Intangible       support=1
  T0010  Phenomenon               <: Thing            support=1
  T0011  Resource                 <: Thing            support=1
  T0012  TemperatureRange         <: QuantitativeValueDistribution support=1
  T0013  NaturalDisaster          <: Thing            support=1
  T0014  Standard                 <: I

# ─────────────────────────── GLOBAL RE-CONSOLIDATION ───────────────────────────
## Everything below is the global-reconsolidation block

Run the pipeline above (through **Build** and **Report + verify**) to produce the raw incremental
vocabulary (`type_vocabulary<_osfx>.json`), then run this block. Every knob it uses —
`CONSOLIDATE`, `SUBTYPE_MODE`, `CONSOLIDATE_TOP_K`, `CONSOLIDATE_INCLUDE_SCHEMA`,
`RERANK_CONSOLIDATE(_FROM)` — comes from the experiment config, not from a cell here.
Setting `consolidate: false` in the config (the **no-consolidation** ablation) skips the LLM
judgement but still writes both output files, identical to the raw ones, so the chain still runs.

**6e depends on it.** Step 3 below writes the per-article `nary_graph…_typed_consolidated.json`, which
is exactly what 6e reads — skip this block and 6e stops at its `assert ARTICLE_ORDER`. It is also where
duplicate mints get collapsed, and with `INLINE_VOCAB_CANDIDATES = False` the inline stage *cannot*
reuse an earlier mint, so it necessarily produces duplicates for this pass to clean up.

**This block writes only NEW files** — every incremental output (`type_vocabulary<_osfx>.json`, the
per-article `*_typed.json`) is left completely untouched, so you can always compare before/after.

**Candidate pool — `CONSOLIDATE_INCLUDE_SCHEMA`.** Every vocabulary type is embedded and shortlisted
against *one ranked pool* (`CONSOLIDATE_TOP_K` nearest neighbours each); those neighbours are the only
merge candidates the judge may use.

* **`False` (vocab-only)** — the pool is the other extension types. The extension is canonicalized
  against itself and a group's survivor is always another `T…` entry. This is the right setting when
  the inline stage already ran schema.org-only (`INLINE_VOCAB_CANDIDATES = False`): schema.org has had
  its shot, and this pass exists to collapse the duplicate mints that stage necessarily produced.
* **`True` (mixed)** — the pool also holds the full schema.org class pool, so a minted type that merely
  restates a standard class can be **folded into schema.org** (the extension entry disappears, its nodes
  are re-typed) or **parented under it** in `subclass` mode. schema.org classes are fixed: *survivor /
  parent* only, never merged away, so only `T…` ids ever appear in `merge_ids`.

**Reranking — `RERANK_CONSOLIDATE`.** When on, e5 fetches `RERANK_CONSOLIDATE_FROM` neighbours and a
cross-encoder picks the `CONSOLIDATE_TOP_K` actually shown — the same two-stage retrieval Hook A uses,
but at its own depth (this pool is the vocabulary, orders smaller than schema.org, so it needs less). The score
printed beside each candidate stays the embedding cosine — a cross-encoder logit is on another scale.

**Two guards on the schema.org side.** `SCHEMA_EXCLUDE_LABELS` drops schema.org's metamodel classes
(`Property`, `Class`, `Enumeration`, `DataType`) from the pool — they describe the vocabulary itself, and
retrieval otherwise offers `schema:Property` for every "quality / attribute" type, where a merge would
assert that a mention like *"music taste"* IS an rdf:Property. And when a schema.org keep shares its
**label** with one of its children, that is a duplicate rather than a specialisation: the child is merged
into schema.org instead of nested under it, which would otherwise make the entry its own parent (`parent`
holds a bare label, and the vocab side wins the lookup) and silently drop it from the hierarchy. A final
sweep nulls any self-parent that slips through and lists it in the report as `self_parent_fixed`.

**Prompt layout.** One interleaved listing: each vocabulary type's line, then its candidates indented
under it as `->` lines, each carrying its own score, source, parent/path and definition (truncated to
`CANDIDATE_DEF_CHARS`). Nothing is referenced by bare id, so the judge never has to resolve a candidate
against a separate dictionary block — it reads one type and its options together and moves on.

Because the LLM only ever sees each type's shortlist (not the full cross-product), the pass scales with
vocabulary size the same way the incremental build does — the prompt grows as `N × (1 + TOP_K)` lines.

The block, in order (all outputs are separate from the incremental build):
1. **Config + prompt** — the separate output paths, `load_consolidated`, `CONSOLIDATE_ALL_PROMPT` (the knobs themselves come from the experiment config).
2. **`consolidate_vocab`** — mixed-pool shortlist inlined per type + ONE Gemini call → writes `e2e type_vocabulary_consolidated.json` + `type_consolidation_report.json` (raw `type_vocabulary.json` untouched). The report records the full `neighbour_shortlist` (each candidate tagged `vocab` / `schema.org`) and, per merge/re-parent, whether the embedder actually proposed that pair (`in_shortlist`).
3. **`refresh_ext_after_consolidation`** — no LLM; writes a separate `nary_graph_typed_consolidated.json` per article (incremental `*_ext.json` untouched) **plus** a changes-only file `consolidation_ext_changes.json` (before → after of every node that moved, entities **and** Event-hub `nary_nodes`).
4. **`build_type_hierarchy`** — indented schema.org + extension tree, written to TWO separate files: `type_hierarchy.txt` (raw incremental) and `type_hierarchy_consolidated.txt` (consolidated).

In [ ]:
# Global consolidation prompt — used by consolidate_vocab below.
#
# From the experiment config (do NOT reassign them here — that would override the run's own record):
#   CONSOLIDATE                 False -> skip the LLM judgement entirely (the `no-consolidation`
#                               ablation). The *_consolidated files are still written, identical to
#                               the raw ones, so 6e still has its input and the metrics show a real 0.
#   SUBTYPE_MODE                how a SPECIFIC type relates to a broader type already in the pool:
#                                 "off"      only merge exact synonyms (no broader/narrower handling)
#                                 "subclass" KEEP the specific type, set its parent to the broader
#                                            one — builds an is-a hierarchy, preserves granularity
#                                 "merge"    COLLAPSE the specific type into the broader parent
#   CONSOLIDATE_TOP_K           nearest-neighbour shortlist size per item (retrieve-then-judge)
#   CONSOLIDATE_INCLUDE_SCHEMA  which pool each vocab type is shortlisted against:
#                                 True  MIXED — other vocab types u the whole schema.org class pool,
#                                       so a vocab type that duplicates a standard class can be
#                                       folded INTO schema.org (or parented under it). schema.org
#                                       classes are fixed: SURVIVOR / PARENT only, never merged away.
#                                 False VOCAB-ONLY — the extension is canonicalized against itself.
#                                       Use when the inline stage already had its shot at schema.org
#                                       (INLINE_VOCAB_CANDIDATES = False) and this pass exists only
#                                       to collapse the duplicate mints that stage must produce.
assert SUBTYPE_MODE in ("off", "subclass", "merge"), "subtype_mode must be 'off', 'subclass' or 'merge'"
SCHEMA_ID_PREFIX = "schema:"    # how a schema.org candidate is identified to the LLM (e.g. schema:Person)
CANDIDATE_DEF_CHARS = 200       # per-candidate definition budget (candidates are inlined under each type)
# schema.org METAMODEL classes describe the vocabulary itself (rdf:Property / rdfs:Class), not anything
# an article talks about, so they are dropped from the candidate pool. Retrieval otherwise surfaces
# schema:Property for every "quality / attribute" type, and merging into it asserts that e.g. the
# mention "music taste" IS an rdf:Property — a category error. Empty the set to allow them back.
SCHEMA_EXCLUDE_LABELS = {"Property", "Class", "Enumeration", "DataType"}

# Consolidation writes to SEPARATE files — the raw incremental type_vocabulary.json is left INTACT.
CONSOLIDATED_VOCAB_PATH   = RESULTS_DIR / f"_global/e2e/type_vocabulary{_osfx}_consolidated.json"   # the consolidated snapshot
CONSOLIDATION_REPORT_PATH = RESULTS_DIR / f"_global/e2e/type_consolidation_report{_osfx}.json"      # what merged / re-parented
def load_consolidated():
    """The consolidated vocab if it exists, else fall back to the raw incremental one."""
    return json.loads(CONSOLIDATED_VOCAB_PATH.read_text()) if CONSOLIDATED_VOCAB_PATH.exists() else load_vocab()

CONSOLIDATE_ALL_PROMPT = """You are organizing a vocabulary of entity types. Each type is listed with its id, label, parent, support and definition, followed by its CANDIDATES: the nearest other entries. Produce groups of two kinds:

SYNONYM — entries denoting the same concept, differing only in wording. keep_id = the entry that survives; merge_ids = the rest.
SUBTYPE — vocabulary types that are all kinds of one more general type. keep_id = the general parent; merge_ids = its children. Group a child under a parent only if every mention under the child is still a valid instance of the parent.

Rules:
- Group a type only with its own candidates, or with a type whose candidates include it. Never invent an id.
- `T...` ids are vocabulary types minted by this pipeline; only these may appear in merge_ids.
- `schema:X` ids are schema.org classes; they are fixed, so they may appear only as keep_id.
- Prefer schema.org when it fits: an extension type that restates a standard class is redundant. A `schema:X` keep_id works for either kind — SYNONYM when it denotes the same concept, SUBTYPE when it is broader.
- A vocabulary entry with the same label as a schema.org candidate is a duplicate, not a specialisation: SYNONYM, keep_id = `schema:X`.
- Judge by label and definition, not the label alone. When unsure, leave a type ungrouped.
- Each type belongs to at most one group.

Return only valid JSON — no prose, no code fences:
{"groups": [{"kind": "subtype", "keep_id": "<parent id>", "merge_ids": ["<child id>"], "reason": "one short line"}, {"kind": "synonym", "keep_id": "<schema:Class or id>", "merge_ids": ["<id>"], "reason": "one short line"}]}"""
print(f"consolidation prompt loaded | SUBTYPE_MODE={SUBTYPE_MODE!r} | "
      f"mixed pool (schema.org candidates): {CONSOLIDATE_INCLUDE_SCHEMA} | consolidated -> {CONSOLIDATED_VOCAB_PATH.name}")

In [ ]:
def _consolidation_disabled():
    """CONSOLIDATE=False: copy the raw vocabulary to the consolidated path and write an empty
    report, so every consumer downstream is unchanged and the run is still fully described."""
    v = load_vocab()
    CONSOLIDATED_VOCAB_PATH.write_text(json.dumps(v, indent=2, ensure_ascii=False))
    rep = {"timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "consolidation": "DISABLED",
           "experiment": EXPERIMENT, "subtype_mode": SUBTYPE_MODE,
           "source_vocab": VOCAB_PATH.name, "consolidated_vocab": CONSOLIDATED_VOCAB_PATH.name,
           "vocab_size_before": len(v["types"]), "vocab_size_after": len(v["types"]),
           "n_merges": 0, "n_types_removed": 0, "n_reparented": 0,
           "n_merged_into_schema": 0, "n_reparented_under_schema": 0,
           "merges": [], "reparents": [], "neighbour_shortlist": [], "self_parent_fixed": []}
    CONSOLIDATION_REPORT_PATH.write_text(json.dumps(rep, indent=2, ensure_ascii=False))
    print(f"consolidation DISABLED for experiment {EXPERIMENT!r} — copied the raw vocabulary "
          f"({len(v['types'])} types) to {CONSOLIDATED_VOCAB_PATH.name} unchanged")
    return rep


# Retrieve-then-judge: every vocab type is shortlisted against the other vocab types, plus the full
# schema.org class pool when CONSOLIDATE_INCLUDE_SCHEMA is on (then the judge sees standard classes and
# sibling extension types side by side as merge candidates). With RERANK_CONSOLIDATE on, e5 fetches
# RERANK_CONSOLIDATE_FROM neighbours and a cross-encoder picks the CONSOLIDATE_TOP_K shown to the
# judge — the same two-stage retrieval Hook A uses, at its own depth. The score printed for each
# candidate stays the EMBEDDING cosine.
def _candidate_line(src, obj, score):
    """One rendered candidate, inlined under the type it was retrieved for — carries everything the
    judge needs (id, label, parent/path, definition), so no separate lookup block is needed."""
    if src == "schema.org":
        desc = (obj.get("enriched_comment") or obj.get("comment") or "").replace("\n", " ").strip()
        return (f'    -> {SCHEMA_ID_PREFIX}{obj["label"]} ({score}) | schema.org | path: {obj.get("path", "")}'
                f' | def: {desc[:CANDIDATE_DEF_CHARS]}')
    desc = (obj.get("definition") or "").replace("\n", " ").strip()
    return (f'    -> {obj["id"]} ({score}) | vocab | label: {obj["canonical_label"]}'
            f' | parent: {obj.get("parent")} | support: {obj.get("support", 0)}'
            f' | def: {desc[:CANDIDATE_DEF_CHARS]}')

def _consolidation_shortlist(ents_sorted):
    """Top-CONSOLIDATE_TOP_K nearest neighbours of each vocab type over (vocab \\ self) u schema.org.
    Returns (cand_lines, shortlist_records, adj): `cand_lines` maps a vocab id to its rendered candidate
    lines (spliced under that type's line in the prompt), `adj` maps it to the set of candidate ids the
    embedder proposed (used for the audit), `shortlist_records` is the same thing for the report."""
    ids  = [t["id"] for t in ents_sorted]
    labs = [t["canonical_label"] for t in ents_sorted]
    V    = embed([vocab_embedding_text(t) for t in ents_sorted], DP)
    meta = [("vocab", ids[i], labs[i], ents_sorted[i]) for i in range(len(ents_sorted))]
    D    = V
    if CONSOLIDATE_INCLUDE_SCHEMA:
        keep = [i for i, c in enumerate(schema_pool) if c["label"] not in SCHEMA_EXCLUDE_LABELS]
        D    = np.vstack([V, schema_emb[keep]])
        meta = meta + [("schema.org", SCHEMA_ID_PREFIX + schema_pool[i]["label"],
                        schema_pool[i]["label"], schema_pool[i]) for i in keep]
    sims = V @ D.T
    for i in range(len(ents_sorted)):
        sims[i, i] = -1.0                                   # a type is never its own candidate
    k = min(CONSOLIDATE_TOP_K, D.shape[0] - 1)

    def _cand_doc(m):
        """"label: definition" for the cross-encoder — vocab entries and schema.org entries alike."""
        obj = m[3]
        d = obj.get("definition") or obj.get("enriched_comment") or obj.get("comment") or ""
        return f"{m[2]}: {d[:400]}"

    fetch = min(max(k, RERANK_CONSOLIDATE_FROM), D.shape[0] - 1) if RERANK_CONSOLIDATE else k
    cand_lines, shortlist, adj = {}, [], {}
    for i in range(len(ents_sorted)):
        top = np.argsort(-sims[i])[:fetch]
        if RERANK_CONSOLIDATE and len(top) > k:      # cross-encode the shortlist, same as Hook A
            q  = vocab_embedding_text(ents_sorted[i])
            rs = np.asarray(_reranker().predict([(q, _cand_doc(meta[j])) for j in top],
                                                batch_size=64, show_progress_bar=False))
            top = top[np.argsort(-rs)][:k]
        # the score shown to the LLM stays the EMBEDDING cosine — a cross-encoder logit is on a
        # different scale and would read as a confidence it does not carry.
        cands = [(meta[j], round(float(sims[i][j]), 3)) for j in top]
        adj[ids[i]] = {m[1] for m, _ in cands}
        cand_lines[ids[i]] = [_candidate_line(m[0], m[3], s) for m, s in cands]
        shortlist.append({"id": ids[i], "label": labs[i],
                          "neighbours": [{"id": m[1], "label": m[2], "source": m[0], "score": s}
                                         for m, s in cands]})
    return cand_lines, shortlist, adj


def consolidate_vocab(vocab, write_report=True, verbose=True):
    """Global re-consolidation in ONE Gemini call, judged against the MIXED candidate pool (vocab u
    schema.org). The model returns SYNONYM and SUBTYPE groups whose keep_id may be a vocab id OR a
    `schema:X` class; how SUBTYPE (is-a) groups are applied depends on SUBTYPE_MODE:
      "off"      -> ignore them (only synonyms merge).
      "subclass" -> KEEP each child type, set its parent to the broader type (vocab OR schema.org).
      "merge"    -> fold each child type into the broader parent (removed; label -> alias).
    SYNONYM groups merge onto the highest-support member, or onto the schema.org class when the LLM
    kept one (the vocab type is then dropped from the extension entirely). Writes the consolidated
    vocab to CONSOLIDATED_VOCAB_PATH and a report to CONSOLIDATION_REPORT_PATH — the raw
    type_vocabulary.json is NOT modified (pass load_vocab() as `vocab`)."""
    ents  = list(vocab["types"])
    by_id = {t["id"]: t for t in ents}

    def _articles_of(t):
        return [p.get("article") for p in t.get("provenance", []) if p.get("article")]
    articles_before = sorted({a for t in ents for a in _articles_of(t)})

    groups = []
    shortlist, _adj = [], {}     # the embedder's proposals, for the audit report
    if len(ents) >= 2:
        ents_sorted = sorted(ents, key=lambda x: x["id"])
        lines = [f'- id: {t["id"]} | label: {t["canonical_label"]} | parent: {t.get("parent")} '
                 f'| support: {t.get("support", 0)} | def: {t.get("definition", "")}'
                 + (f' | aka: {", ".join(t["aliases"])}' if t.get("aliases") else "")
                 for t in ents_sorted]
        cand_lines, shortlist, _adj = _consolidation_shortlist(ents_sorted)
        # Each vocab type is followed by its own candidates (vocab and schema.org ranked together),
        # so every id the judge may reach for is defined right where it is offered. The prompt always
        # asks for BOTH group kinds; with SUBTYPE_MODE = "off" the subtype groups it returns are
        # dropped below rather than suppressed in the prompt.
        blocks = [ln + "\n" + "\n".join(cand_lines[t["id"]]) for t, ln in zip(ents_sorted, lines)]
        msg = ("VOCABULARY — each type, then its retrieved CANDIDATES (`->` lines: the only entries it "
               "may be grouped with):\n\n" + "\n\n".join(blocks))
        try:
            res = parse_json(gemini_json(CONSOLIDATE_ALL_PROMPT, msg))
            groups = (res.get("groups", res) if isinstance(res, dict) else res) or []
        except Exception as ex:
            print(f"[WARN] consolidation call failed ({type(ex).__name__}: {ex})")

    def _keep_ref(keep):
        """Resolve a keep_id to ("vocab", entry) or ("schema.org", label); (None, None) if unknown.
        A bare schema.org label (no `schema:` prefix) is tolerated."""
        if not isinstance(keep, str):
            return None, None
        if keep in by_id:
            return "vocab", by_id[keep]
        lbl = keep[len(SCHEMA_ID_PREFIX):] if keep.startswith(SCHEMA_ID_PREFIX) else keep
        if lbl in SCHEMA_LABELS and lbl not in SCHEMA_EXCLUDE_LABELS:   # excluded ones are never valid keeps
            return "schema.org", lbl
        return None, None

    def _split_label_collisions(groups):
        """A schema.org keep that shares its LABEL with one of its children is not a hierarchy — it is
        the same name twice. `parent` stores a bare label, so attaching the child there would make it
        its OWN parent (self-loop: _find_vocab_by_label resolves the label back to the child, and the
        entry drops out of the hierarchy). An identical label means equivalence, so those children are
        rewritten into a SYNONYM merge into schema.org; any remaining children keep the original kind."""
        out = []
        for g in groups:
            ksrc, kobj = _keep_ref(g.get("keep_id"))
            ids  = list(dict.fromkeys(g.get("merge_ids") or []))
            same = [i for i in ids if ksrc == "schema.org" and i in by_id
                    and by_id[i]["canonical_label"] == kobj]
            if not same:
                out.append(g); continue
            note = " [same label as the schema.org entry -> merged, not nested under it]"
            out.append({**g, "kind": "synonym", "merge_ids": same,
                        "reason": (g.get("reason") or "") + note})
            rest = [i for i in ids if i not in same]
            if rest:
                out.append({**g, "merge_ids": rest})
        return out
    groups = _split_label_collisions(groups)

    merges, reparents, merged_ids = [], [], set()
    label_remap = {}          # merged-away label -> survivor label (to repair `parent` pointers)
    for g in groups:
        kind = (g.get("kind") or "synonym").lower()
        is_subtype = kind.startswith("subtype")
        ksrc, kobj = _keep_ref(g.get("keep_id"))
        if ksrc is None or (ksrc == "vocab" and kobj["id"] in merged_ids):
            continue
        # Only VOCAB types can be merged away — a schema.org id in merge_ids is ignored by design.
        children = [by_id[i] for i in dict.fromkeys(g.get("merge_ids") or [])
                    if i in by_id and i not in merged_ids and not (ksrc == "vocab" and i == kobj["id"])]
        if not children:
            continue

        # SUBTYPE + subclass: keep the children, just re-parent them onto the broader type
        if is_subtype and SUBTYPE_MODE == "subclass":
            parent_label = kobj["canonical_label"] if ksrc == "vocab" else kobj
            rec = {"kind": "subtype", "action": "subclass",
                   "parent_id": kobj["id"] if ksrc == "vocab" else SCHEMA_ID_PREFIX + kobj,
                   "parent_label": parent_label, "parent_source": ksrc,
                   "reason": g.get("reason"), "children": []}
            for child in children:
                if child.get("parent") == parent_label:
                    continue                                      # already under this parent — no-op, skip
                rec["children"].append({"id": child["id"], "label": child["canonical_label"],
                                        "old_parent": child.get("parent")})
                child["parent"] = parent_label                    # keep the child type; re-parent it
            if rec["children"]:
                reparents.append(rec)
            continue

        # otherwise MERGE: synonyms always; subtype only when SUBTYPE_MODE == "merge"
        if is_subtype and SUBTYPE_MODE != "merge":
            continue
        if ksrc == "schema.org":
            survivor, members = None, children                    # folds INTO schema.org: nothing survives in vocab
            survivor_id, survivor_label = SCHEMA_ID_PREFIX + kobj, kobj
        elif is_subtype:
            survivor, members = kobj, children                    # broader parent survives
        else:
            members  = [kobj] + children
            survivor = max(members, key=lambda t: (t.get("support", 0), -int(t["id"][1:])))
        if survivor is not None:
            survivor_id, survivor_label = survivor["id"], survivor["canonical_label"]
        rec = {"kind": kind, "action": "merge", "survivor_id": survivor_id,
               "survivor_label": survivor_label,
               "survivor_source": "schema.org" if survivor is None else "vocab",
               "reason": g.get("reason"), "merged": [],
               "articles": sorted({a for t in members for a in _articles_of(t)})}
        for m in members:
            if survivor is not None and m["id"] == survivor["id"]:
                continue
            if survivor is not None:                              # schema.org survivors hold no aliases/provenance
                for lbl in [m["canonical_label"]] + m.get("aliases", []):
                    if lbl != survivor["canonical_label"] and lbl not in survivor["aliases"]:
                        survivor["aliases"].append(lbl)
                survivor["support"] = survivor.get("support", 0) + m.get("support", 0)
                survivor.setdefault("provenance", []).extend(m.get("provenance", []))
            rec["merged"].append({"id": m["id"], "label": m["canonical_label"],
                                  "support": m.get("support", 0), "articles": _articles_of(m)})
            merged_ids.add(m["id"])
            label_remap[m["canonical_label"]] = survivor_label
        if rec["merged"]:
            if survivor is not None:
                rec["resulting_support"]  = survivor["support"]
                rec["resulting_aliases"] = list(survivor["aliases"])
            merges.append(rec)

    if merged_ids:
        vocab["types"] = [t for t in vocab["types"] if t["id"] not in merged_ids]
    # A merged-away label can still be some surviving type's `parent` — repoint it at the survivor
    # (following merge chains), so no parent link dangles after consolidation.
    for t in vocab["types"]:
        p, seen = t.get("parent"), set()
        while p in label_remap and p not in seen:
            seen.add(p); p = label_remap[p]
        if p != t.get("parent"):
            t["parent"] = p
    # Safety net: an entry must never be its own parent. A bare label can name both a vocab entry and a
    # schema.org one, and the vocab side wins the lookup — so a self-label parent silently orphans the
    # entry from the hierarchy. Drop it to no super-entry and record it in the report.
    self_parented = [t["id"] for t in vocab["types"] if t.get("parent") == t["canonical_label"]]
    for t in vocab["types"]:
        if t.get("parent") == t["canonical_label"]:
            t["parent"] = None

    # write the CONSOLIDATED vocabulary to its own file (raw type_vocabulary.json stays untouched)
    _annotate_vocab_paths(vocab)                 # refresh derived `path`s (re-parenting/merges changed them)
    CONSOLIDATED_VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

    n_reparented   = sum(len(r["children"]) for r in reparents)
    n_into_schema  = sum(len(r["merged"]) for r in merges if r["survivor_source"] == "schema.org")
    n_sch_parents  = sum(len(r["children"]) for r in reparents if r["parent_source"] == "schema.org")
    # audit: did the embedder's shortlist actually propose each pair the LLM merged/re-parented?
    def _in_shortlist(a, b):
        return b in _adj.get(a, ()) or a in _adj.get(b, ())
    for _r in merges:
        for _m in _r["merged"]:
            _m["in_shortlist"] = _in_shortlist(_r["survivor_id"], _m["id"])
    for _r in reparents:
        for _c in _r["children"]:
            _c["in_shortlist"] = _in_shortlist(_r["parent_id"], _c["id"])

    report = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "experiment": EXPERIMENT,
        "match_model": MATCH_MODEL,
        "gemini_model": GEMINI_MODEL, "expand_mode": EXPAND_MODE,
        "subtype_mode": SUBTYPE_MODE, "source_vocab": str(VOCAB_PATH.name),
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "consolidate_top_k": CONSOLIDATE_TOP_K,
        "candidate_pool": ("vocab+schema.org" if CONSOLIDATE_INCLUDE_SCHEMA else "vocab"),
        "consolidation_reranked": RERANK_CONSOLIDATE,
        "schema_excluded_labels": sorted(SCHEMA_EXCLUDE_LABELS), "self_parent_fixed": self_parented,
        "neighbour_shortlist": shortlist,
        "articles_used": articles_before, "n_articles": len(articles_before),
        "vocab_size_before": len(ents), "vocab_size_after": len(vocab["types"]),
        "n_merges": len(merges), "n_types_removed": len(merged_ids), "n_reparented": n_reparented,
        "n_merged_into_schema": n_into_schema, "n_reparented_under_schema": n_sch_parents,
        "merges": merges, "reparents": reparents,
    }
    if write_report:
        CONSOLIDATION_REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False))

    if verbose:
        print(f"consolidation (1 call): subtype_mode={SUBTYPE_MODE!r} | pool={report['candidate_pool']} | "
              f"{len(merges)} merge-group(s), {len(merged_ids)} removed "
              f"({n_into_schema} into schema.org), {n_reparented} re-parented "
              f"({n_sch_parents} under schema.org) | vocab {len(ents)} -> {len(vocab['types'])}")
        for r in merges:
            tag = "schema.org" if r["survivor_source"] == "schema.org" else r["survivor_id"]
            print(f"  [merge/{r['kind'][:4]}] keep {tag} {r['survivor_label']!r} <= "
                  f"{[m['label'] for m in r['merged']]}")
        for r in reparents:
            tag = " (schema.org)" if r["parent_source"] == "schema.org" else ""
            print(f"  [subtype] {[c['label'] for c in r['children']]} <: {r['parent_label']}{tag}")
        print(f"  consolidated vocab -> {CONSOLIDATED_VOCAB_PATH}  (raw {VOCAB_PATH.name} untouched)")
        if write_report:
            print(f"  report -> {CONSOLIDATION_REPORT_PATH}")
    return report

# Runs on the RAW incremental vocab and writes the consolidated result to a separate file.
# CONSOLIDATE=False (the `no-consolidation` ablation) skips the LLM but still writes both files, so
# the downstream stages — which read the *_consolidated names — behave identically and the metrics
# record a genuine zero instead of a missing file.
_ = consolidate_vocab(load_vocab()) if CONSOLIDATE else _consolidation_disabled()

### Refresh `*_ext.json` after consolidation → **separate** files — no LLM

Apply the consolidation's `merged_id → survivor` mapping (and subclass re-parentings) to the per-article
files deterministically, but write the result to a **new** `nary_graph_typed_consolidated.json`
per article — the incremental `*_ext.json` is left untouched. Both `entities` and the Event-hub
`nary_nodes` are refreshed. When a type merged **into schema.org**, the node stops being an extension
type: `type_source` flips to `"schema.org"` and `type_vocab_id` / `type_definition` / `type_parent`
go null, exactly like a node the build had typed with schema.org directly. Also emit one global
`consolidation_ext_changes.json` that lists **only** the nodes that changed (before → after). Cheaper and
more faithful than `FRESH=True`: it keeps the original type decisions and only reflects the merges /
re-parentings. Run this right after `consolidate_vocab`.

In [56]:
# Produce a CONSOLIDATED copy of each per-article *_ext.json WITHOUT the LLM — the incremental
# *_ext.json files are left UNTOUCHED. Three effects, applied only to the copy:
#   - MERGED types: nodes pointing at a removed vocab_id are remapped to its survivor.
#   - MERGED INTO SCHEMA.ORG: when the survivor is a schema.org class the node stops being an
#     extension type altogether — type_source becomes "schema.org", type_vocab_id/definition/parent null.
#   - RE-PARENTED types (subclass mode): the type survives, so we re-sync every vocab-typed node's
#     label / definition / parent from the CONSOLIDATED vocabulary (picks up the new parent).
# Both `entities` AND `nary_nodes` are refreshed (the build canonicalizes types for both).
# `type_is_new_to_vocab` is a BUILD-TIME provenance fact (was the type newly minted when this article
# was processed) — consolidation leaves it untouched, so only genuine merges / re-parentings register
# as changes. Writes, per article, nary_graph_typed_consolidated.json, and ONE global changes
# file (results/consolidation_ext_changes.json) listing ONLY the nodes that changed (before -> after).
# Pure bookkeeping — type DECISIONS are untouched. Run right after consolidate_vocab.
EXT_CONSOLIDATED_NAME = f"nary_graph{_osfx}_typed_consolidated.json"   # per-article refreshed copy
EXT_CHANGES_PATH      = RESULTS_DIR / f"_global/e2e/consolidation_ext_changes{_osfx}.json"   # global "what changed" file
_CHANGED_FIELDS = ("type_vocab_id", "type_source", "type", "type_definition", "type_parent")   # substantive fields only

def refresh_ext_after_consolidation(report=None, verbose=True):
    if report is None:
        if not CONSOLIDATION_REPORT_PATH.exists():
            print("no consolidation report found — run consolidate_vocab first"); return
        report = json.loads(CONSOLIDATION_REPORT_PATH.read_text())
    # merged vocab id -> where it went: another vocab id, or a schema.org class label
    remap = {}
    for r in report.get("merges", []):
        tgt = ({"source": "schema.org", "label": r["survivor_label"]}
               if r.get("survivor_source") == "schema.org" else
               {"source": "vocab", "id": r["survivor_id"]})
        for m in r.get("merged", []):
            remap[m["id"]] = tgt
    by_id = {t["id"]: t for t in load_consolidated()["types"]}    # survivors + re-parented types

    def _resolve(v):
        """The consolidated fields for a node currently typed with vocab id `v` (None: leave as-is).
        Follows merge chains (a survivor of one group can be merged again by a later one)."""
        tgt, seen = remap.get(v), set()
        while tgt is not None and tgt["source"] == "vocab" and tgt["id"] in remap and tgt["id"] not in seen:
            seen.add(tgt["id"]); tgt = remap[tgt["id"]]
        if tgt is not None and tgt["source"] == "schema.org":     # left the extension for schema.org
            return {"type_vocab_id": None, "type_source": "schema.org", "type": tgt["label"],
                    "type_definition": None, "type_parent": None}
        s = by_id.get(tgt["id"] if tgt else v)                    # follow a merge, else same id
        if s is None:
            return None
        return {"type_vocab_id": s["id"], "type_source": "vocab", "type": s["canonical_label"],
                "type_definition": s.get("definition"), "type_parent": s.get("parent")}

    all_changes, total = [], 0
    for fp in sorted(RESULTS_DIR.glob(f"*/e2e/nary_graph{_osfx}_typed.json")):
        data = json.loads(fp.read_text())                        # the untouched incremental file
        aid  = data.get("article_id", fp.parent.name)
        art_changes = []
        for n in data.get("entities", []) + data.get("nary_nodes", []):
            v = n.get("type_vocab_id")
            if not v:
                continue
            new = _resolve(v)
            if new is None:
                continue
            old = {k: n.get(k) for k in _CHANGED_FIELDS}
            if new != old:
                n.update(new)                                     # mutate the in-memory COPY only
                art_changes.append({"node_id": n.get("id"), "text": n.get("text") or n.get("label"),
                                    "type_oie": n.get("type_oie"), "before": old, "after": new})
        # write the consolidated copy for EVERY article (the incremental *_ext.json stays intact)
        data["consolidated"], data["subtype_mode"] = True, SUBTYPE_MODE
        (fp.parent / EXT_CONSOLIDATED_NAME).write_text(json.dumps(data, indent=2, ensure_ascii=False))
        if art_changes:
            all_changes.append({"article_id": aid, "n_changed": len(art_changes), "changes": art_changes})
            total += len(art_changes)
            if verbose:
                print(f"  {aid}: {len(art_changes)} node(s) changed")

    EXT_CHANGES_PATH.write_text(json.dumps({
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "subtype_mode": SUBTYPE_MODE,
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "n_articles_changed": len(all_changes), "n_nodes_changed": total, "articles": all_changes,
    }, indent=2, ensure_ascii=False))
    print(f"done — wrote {EXT_CONSOLIDATED_NAME} for each article (incremental *_ext.json untouched)\n"
          f"       {total} changed node(s) across {len(all_changes)} article(s) -> {EXT_CHANGES_PATH.name}")

refresh_ext_after_consolidation()

  art1: 10 node(s) changed
  art2: 14 node(s) changed
  art4: 4 node(s) changed
  art5: 8 node(s) changed
done — wrote nary_graph_dedup_schemainline_rr_typed_consolidated.json for each article (incremental *_ext.json untouched)
       36 changed node(s) across 4 article(s) -> consolidation_ext_changes_dedup_schemainline_rr.json


In [ ]:
# ---- PERIODIC RE-CONSOLIDATION (CONSOLIDATE_EVERY) -------------------------------------------
# Off by default: with CONSOLIDATE_EVERY = 0 this cell does nothing and the stage is the classic
# "build everything, consolidate once at the end".
#
# With it set to N, the vocabulary is re-consolidated every N articles and the merged result is fed
# BACK as the working vocabulary, so article N+1 builds on the collapsed term set rather than on
# every duplicate produced so far. That is the point of the knob: it bounds both the vocabulary and
# the consolidation prompt, which is the largest in the pipeline (~25-40k tokens) and grows with the
# vocabulary — on a 59-article corpus the single end-of-run call is what would hit the context limit.
#
# Two things this has to get right, and why it is not just "call consolidate_vocab in a loop":
#
#   ids     the consolidated vocabulary keeps `next_id` (85 types -> 75 survivors, next_id still 86),
#           so feeding it back never re-issues a retired id.
#   remap   refresh_ext_after_consolidation builds its mapping from ONE report and follows merge
#           chains inside it. Each pass overwrites the report, so refreshing after every pass would
#           lose the earlier ones and leave article 1 pointing at a term retired in pass 1. Instead
#           the passes are ACCUMULATED into one cumulative report and the graphs are refreshed once,
#           at the end — the chain-following in _resolve then composes the passes for free
#           (T0003 -> T0001 in pass 1, T0001 -> T0007 in pass 2 resolves to T0007).
#
# The cumulative report is also what run_metrics reads, so cons_merge_groups / cons_merged_into_schema
# are the totals across the whole stage rather than the last pass only.
if CONSOLIDATE_EVERY and len(TODO_ALL) > len(todo):
    _reports = [json.loads(CONSOLIDATION_REPORT_PATH.read_text())] if CONSOLIDATION_REPORT_PATH.exists() else []
    _rest = TODO_ALL[len(todo):]
    _chunks = [_rest[i:i + CONSOLIDATE_EVERY] for i in range(0, len(_rest), CONSOLIDATE_EVERY)]
    print(f"periodic consolidation: {len(_chunks)} further chunk(s) of <= {CONSOLIDATE_EVERY} article(s)\n")

    for _ci, _chunk in enumerate(_chunks, start=2):
        # feed the consolidated vocabulary back in — without this the raw vocabulary keeps every
        # duplicate and the periodic passes are mere checkpoints.
        # RECONCILING feed-back. A pass must only ever remove a type it RECORDED as merged;
        # anything else that disappears is a term the per-article graphs still point at, and feeding
        # a lossy vocabulary forward turns those references into dangling ids that no later pass can
        # repair (it no longer knows the term existed). Measured on the first periodic 6e run: 5 of
        # 29 removals had no merge record, orphaning 9 relations. Restore them and say so, rather
        # than carrying the loss silently into the next chunk.
        _prev = {t["id"]: t for t in load_vocab()["types"]}
        _con = load_consolidated()
        _merged = {m["id"] for g in (_reports[-1].get("merges") or []) for m in (g.get("merged") or [])}
        _lost = [i for i in _prev if i not in {t["id"] for t in _con["types"]} and i not in _merged]
        if _lost:
            print(f"  [repair] {len(_lost)} type(s) left the vocabulary with no merge record "
                  f"-- restoring: {_lost}")
            _con["types"] = _con["types"] + [_prev[i] for i in _lost]
        save_vocab(_con)
        vocab = load_vocab()
        print(f"── chunk {_ci}: {len(vocab['types'])} type(s) carried forward | {_chunk}")
        _cpre = {}
        if PARALLEL_LLM and not INLINE_VOCAB_CANDIDATES and len(_chunk) > 1:
            _cpre = {r.item: r.value for r in
                     nb_parallel.pmap(lambda a: prefetch_types(a, vocab), _chunk, workers=WORKERS)
                     if r.ok}
        for aid in _chunk:
            with nb_parallel.usage_scope(aid):
                canonicalize_article_types(aid, vocab, write=True, save=True, verbose=True,
                                           pre=_cpre.get(aid))
            if aid not in _cpre:
                time.sleep(5)
        if CONSOLIDATE:
            _reports.append(consolidate_vocab(load_vocab()))
        else:
            _consolidation_disabled()

    # One cumulative report: the union of every pass, so the single refresh below composes them.
    if _reports:
        _cum = dict(_reports[-1])
        _cum["merges"] = [g for r in _reports for g in (r.get("merges") or [])]
        _cum["reparents"] = [g for r in _reports for g in (r.get("reparents") or [])]
        _cum["consolidation_passes"] = len(_reports)
        _cum["n_merges"] = len(_cum["merges"])
        _cum["n_reparented"] = sum(len(g.get("children") or []) for g in _cum["reparents"])
        _cum["n_types_removed"] = sum(len(g.get("merged") or []) for g in _cum["merges"])
        _cum["n_merged_into_schema"] = sum(len(g.get("merged") or []) for g in _cum["merges"]
                                           if g.get("survivor_source") == "schema.org")
        # NOT pass 1's "before" and pass 4's "after" — those describe different populations (the
        # vocabulary after one chunk vs after twenty), and quoted as a pair they read as though
        # consolidation GREW the vocabulary. The meaningful pair for a periodic run is the
        # counterfactual: what the vocabulary would be had nothing been collapsed, against what it
        # actually is. (Sanity check: before == next_id - 1, i.e. every id ever issued.)
        _cum["vocab_size_after"] = _reports[-1].get("vocab_size_after")
        _cum["vocab_size_before"] = (_cum["vocab_size_after"] or 0) + _cum["n_types_removed"]
        _cum["per_pass"] = [{"before": r.get("vocab_size_before"), "after": r.get("vocab_size_after")}
                            for r in _reports]
        CONSOLIDATION_REPORT_PATH.write_text(json.dumps(_cum, indent=2, ensure_ascii=False))
        print(f"\ncumulative report over {len(_reports)} pass(es) -> {CONSOLIDATION_REPORT_PATH.name}"
              f"   {_cum['n_merges']} merge group(s), {_cum['n_types_removed']} type(s) removed")

    refresh_ext_after_consolidation()          # once, with the composed mapping
    print(f"\ndone — {len(load_consolidated()['types'])} consolidated types over "
          f"{len(TODO_ALL)} article(s)")

# ---- INVARIANT: no consolidated graph may reference a term the vocabulary no longer holds -------
# The risk of consolidating repeatedly is that a term merged in one pass and merged AGAIN in a later
# one loses its trail, leaving per-article graphs pointing at ids that no longer exist. That failure
# is SILENT — the run finishes, the counts look plausible, and only a later audit finds it. Assert it
# here, where it costs nothing and fails loudly.
import glob as _glob
_live = {t["id"] for t in load_consolidated()["types"]}
_dangling = []
for _f in sorted(_glob.glob(str(RESULTS_DIR / "art*" / "e2e" / "*_typed_consolidated.json"))):
    _g = json.loads(Path(_f).read_text())
    for _coll in ("entities", "nary_nodes"):
        for _n in _g.get(_coll, []):
            _v = _n.get("type_vocab_id")
            if _v and _v not in _live:
                _dangling.append((Path(_f).parts[-3], _v, _n.get("type")))
if _dangling:
    _ids = {d[1] for d in _dangling}
    print(f"\n  *** {len(_dangling)} reference(s) to {len(_ids)} vanished type(s):")
    for _d in _dangling[:10]:
        print(f"      {_d[0]}  {_d[1]}  {_d[2]}")
    raise AssertionError(
        f"{len(_dangling)} graph reference(s) point at a type missing from the consolidated "
        f"vocabulary — these graphs are NOT safe to report. Fix before using this run.")
print("  invariant OK — every vocabulary reference in the consolidated graphs resolves")


## Type hierarchy (schema.org + extension) — raw and consolidated, **separate** files

One indented tree of the whole type space — schema.org classes and the extension (minted) types
together, each tagged `[schema.org]` or `[extension]`, subclasses nested under their parent. Built
**twice**, into separate files so you can compare: `results/type_hierarchy.txt` from the **raw
incremental** vocab, and `results/type_hierarchy_consolidated.txt` from the **consolidated** vocab (so
`subclass` re-parentings and merges appear). Set `INCLUDE_ALL_SCHEMA=True` to include the entire
schema.org tree, or leave it `False` to show just the branches your extension types hang off.

In [ ]:
# Type hierarchy: schema.org classes + extension (vocab) types in ONE indented tree, each tagged
# [schema.org] or [extension]; children (subclasses) are nested under their parent. Parameterized by
# which vocab to draw the extension types from, so the RAW incremental vocab and the CONSOLIDATED vocab
# each get their own separate file:
#   type_hierarchy.txt              <- raw incremental vocab (type_vocabulary.json)
#   type_hierarchy_consolidated.txt <- consolidated vocab   (re-parentings/merges reflected)
# INCLUDE_ALL_SCHEMA comes from the experiment config:
#   False -> only the schema.org branches that contain an extension type (the ancestors of the
#            minted types) plus every extension type.
#   True  -> the entire schema.org tree as well (large: ~900+ lines).
HIERARCHY_PATH              = RESULTS_DIR / f"_global/e2e/type_hierarchy{_osfx}.txt"                # raw incremental vocab
HIERARCHY_CONSOLIDATED_PATH = RESULTS_DIR / f"_global/e2e/type_hierarchy{_osfx}_consolidated.txt"   # consolidated vocab

def build_type_hierarchy(vocab, write_path, include_all_schema=INCLUDE_ALL_SCHEMA, show=50):
    from collections import defaultdict
    schema_labels = {c["label"] for c in schema_pool}
    schema_parent = {c["label"]: (c["path"].split(" > ")[-1] if c.get("path") else None) for c in schema_pool}
    ext = {t["canonical_label"]: t for t in vocab["types"]}

    parent_of, kind = {}, {}
    for lbl in schema_labels:
        parent_of[lbl] = schema_parent.get(lbl); kind[lbl] = "schema.org"
    for lbl, t in ext.items():                                   # extension overrides if same label
        parent_of[lbl] = t.get("parent"); kind[lbl] = "extension"

    if include_all_schema:
        included = set(schema_labels) | set(ext)
    else:                                                        # only extensions + their ancestor chains
        included = set(ext)
        for lbl in list(ext):
            cur, seen = parent_of.get(lbl), set()
            while cur and cur not in seen and cur not in included:
                seen.add(cur); included.add(cur); cur = parent_of.get(cur)

    children, roots = defaultdict(list), []
    for lbl in included:
        p = parent_of.get(lbl)
        (children[p].append(lbl) if p in included else roots.append(lbl))
    for p in children:
        children[p].sort()

    lines = []
    def walk(lbl, depth, seen):
        tag   = "[extension]" if kind.get(lbl) == "extension" else "[schema.org]"
        extra = f"  (support {ext[lbl].get('support', 0)})" if lbl in ext else ""
        lines.append("  " * depth + f"{lbl}  {tag}{extra}")
        for c in children.get(lbl, []):
            if c not in seen:
                walk(c, depth + 1, seen | {c})
    for r in sorted(set(roots)):
        walk(r, 0, {r})

    text = "\n".join(lines)
    if write_path:
        write_path.write_text(text)
    print(f"type hierarchy: {len(lines)} nodes ({len(ext)} extension) | "
          f"include_all_schema={include_all_schema} -> {write_path}\n" + "-" * 74)
    print(text if len(lines) <= show else "\n".join(lines[:show]) + f"\n  ... ({len(lines) - show} more — see file)")
    return text

# Raw incremental hierarchy (always available), then the consolidated one (reflects re-parentings/merges).
print("=== RAW incremental vocabulary ===")
build_type_hierarchy(load_vocab(),        HIERARCHY_PATH)
print("\n=== CONSOLIDATED vocabulary ===")
build_type_hierarchy(load_consolidated(), HIERARCHY_CONSOLIDATED_PATH)

## Run metrics — every number this stage produced, in one file

`run_metrics.types` re-reads the review files, the typed graphs, the grown vocabulary and the
consolidation report, and reports what the write-up needs: how many distinct free-form types the
corpus produced, the **reuse-vs-mint split** (both per type and per entity), the **retrieval
quality** that split depends on (was the chosen label even in the shortlist, and at which rank),
the vocabulary size **before and after** consolidation, what consolidation did (merges, folds into
schema.org, re-parentings, and the `in_shortlist` audit: how much of it the embedder actually
proposed), and the token cost.

Saved to `_global/e2e/metrics_types<_osfx>.json`, so the numbers outlive the kernel and
`run_metrics.compare(...)` can line several experiments up afterwards.

In [ ]:
# Collect + persist this stage's numbers. Pure reading — no LLM call, safe to re-run, and it works
# on a run that finished days ago.
import importlib, run_metrics
importlib.reload(run_metrics)

_m = run_metrics.types(RESULTS_DIR, ARTICLE_ORDER, isfx=_sfx, osfx=_osfx,
                       track="e2e", experiment=EXPERIMENT)
run_metrics.show(_m)

# save() also CLOSES this stage in the run manifest — the record of WHICH run these numbers
# came from: the resolved config, the git commit, the article order, every output file and
# the token spend.  results/<model>/_runs/<run_id>.json  ·  python -m run_manifest list
print("\nsaved ->", run_metrics.save(_m, RESULTS_DIR, _osfx, "e2e",
                                     section="6d", articles=ARTICLE_ORDER))

# Side by side with other experiments that have already been run and saved (missing ones are
# skipped, so this is safe before the ablations exist):
# print(run_metrics.compare(RESULTS_DIR, "types",
#                           ["_dedup_main", "_dedup_no_rerank", "_dedup_inline_vocab"], "e2e"))